In [ ]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import clear_output
import os
%pip install kagglehub catboost xgboost tqdm -q

clear_output()
from catboost import CatBoostClassifier

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
pathh = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(pathh)

In [ ]:
# Task 2: Write your code here:
print("First 5 rows:")
print(df.head())

In [ ]:
# Task 3: Write your code here:# Display dataset information
print("\nDataset info:")
print(df.info())

In [ ]:
# Task 4: Write your code here:
print("\nStatistical description:")
print(df.describe())

In [ ]:
# Task 1: Write your code here:
print("\nMissing values:")
print(df.isnull().sum())

df = df.fillna(df.median(numeric_only=True))


In [ ]:
# Task 2: Write your code here:
print(f"\nNumber of duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
#I made it so i dont need to do this part


In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()
numerical_cols = df.select_dtypes(include=[np.number]).columns
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [ ]:
# Task 5: Write your code here:

print(f"\nTarget distribution:\n{df['Target'].value_counts()}")
print(f"Target imbalance: {'Yes' if df['Target'].value_counts().max() / df['Target'].value_counts().sum() > 0.7 else 'No'}")

In [ ]:
# Task 1: Write your code here:

X = df.drop('Target', axis=1)
y = df['Target'].astype(int)


In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(random_state=42, verbose=0)

accuracies = []
f1_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    accuracies.append(accuracy_score(y_val, y_pred))
    f1_scores.append(f1_score(y_val, y_pred))

print(f"\nAverage Accuracy: {np.mean(accuracies):.4f}")
print(f"Average F1 Score: {np.mean(f1_scores):.4f}")


In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 Feature Importances:")
print(feature_importance.head(5))

In [ ]:

coeffs = {}

coeffs['Lasso'] = model['LASSO Regression'].coef_
coeffs['Ridge'] = model['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden_feature = df.iloc['feature']
print(f"\nGolden Feature: {golden_feature}")

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()



In [ ]:
# Task Bonus: Write your code here:
X_golden = df[[golden_feature]]

# Retrain with only golden feature
accuracies_golden = []
f1_scores_golden = []

for train_idx, val_idx in skf.split(X_golden, y):
    X_train_golden, X_val_golden = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_train_golden, y_train)
    y_pred = model.predict(X_val_golden)

    accuracies_golden.append(accuracy_score(y_val, y_pred))
    f1_scores_golden.append(f1_score(y_val, y_pred))

print(f"\nAverage Accuracy (Golden Feature Only): {np.mean(accuracies_golden):.4f}")
print(f"Average F1 Score (Golden Feature Only): {np.mean(f1_scores_golden):.4f}")
